# LSMT barycenters: single-start optimization and runtime benchmarks
Use a GPU runtime for CUDA. Run benchmark cells independently to avoid competing workloads. The fast preset is an ablation starting point, not a claim of globally optimal convergence or speed. See FAST_EXPERIMENT.md for timing and selection definitions.

In [ ]:
%cd /content
!test -d LSOT_GMM || git clone https://github.com/tmp0810/LSOT_GMM.git
%cd /content/LSOT_GMM
!git pull --ff-only origin main
!pip -q install -e .
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Full synthetic 7x7 grids
One screened start, Adam 200, learning rate 0.003, L=100, validation L=512. Add `images` to `--cases` for the four original notebook shapes. Set `--sampling fresh` for fresh directions at every update and use a different output folder.

In [ ]:
!python -m experiments.barycenter.run \
    --config experiments/barycenter/configs/fast.json \
    --cases synthetic --device cuda \
    --output-dir results/barycenter_fast

In [ ]:
from pathlib import Path
from IPython.display import display, Image
for path in sorted(Path("results/barycenter_fast/synthetic").glob("*/barycenters.png")):
    print(path)
    display(Image(filename=str(path)))

## Controlled optimizer ablation
The original MW2 multi-marginal solver and MW2-variational are included. All seeds are reported. `--patience 0` keeps the full update budget. Test losses use independent projections.

In [ ]:
!python -m experiments.barycenter.benchmark --task ablation \
    --cases synthetic --kinds Mix SMix B B1D --seeds 0 1 2 \
    --steps 200 --L 100 --patience 0 --device cuda \
    --output-dir results/barycenter_ablation

## Pairwise transport runtime versus K
This isolates the lifted-plan versus LP comparison. Both total computation and cached-ground-cost timings are saved, including coupling quality and marginal errors.

In [ ]:
!python -m experiments.barycenter.benchmark --task plans \
    --component-counts 5 10 20 50 100 200 500 --kinds Mix SMix B B1D \
    --seeds 0 1 2 --L 100 --repeats 10 --device cuda \
    --output-dir results/lifted_plan_scaling

## Four-input barycenter scaling
New random SPD GMMs, K components per input and Kb=4K-3. This is separate from the original Figure 4 data. Large LPs exceeding the tuple budget are explicitly marked skipped.

In [ ]:
!python -m experiments.barycenter.benchmark --task scaling \
    --component-counts 3 5 8 12 20 --kinds B1D \
    --variants fast-fixed fast-fresh --seeds 0 1 2 \
    --steps 200 --patience 0 --device cuda \
    --output-dir results/barycenter_scaling